## Neural Radiance Fields

In [ ]:
import io
import os

import imageio
import numpy as np
import torch
from IPython.display import display, Image

from models import PositionalEmbeddings, NeRF
from utils.render_utils import render_path

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    embed_pts = PositionalEmbeddings(
        **checkpoint["embed_pts"],
        periodic_functions=(torch.sin, torch.cos),
    )

    embed_viewdirs = None
    if checkpoint["embed_viewdirs"] is not None:
        embed_viewdirs = PositionalEmbeddings(
            **checkpoint["embed_viewdirs"],
            periodic_functions=(torch.sin, torch.cos),
        )

    nerf_coarse_cfg = checkpoint["nerf_coarse"]
    nerf_coarse = NeRF(
        depth=nerf_coarse_cfg["depth"],
        W=nerf_coarse_cfg["W"],
        in_channels=nerf_coarse_cfg["in_channels"],
        in_channel_views=nerf_coarse_cfg["in_channel_views"],
        out_channels=nerf_coarse_cfg["out_channels"],
        skip_connections=nerf_coarse_cfg["skip_connections"],
        use_viewdirs=nerf_coarse_cfg["use_viewdirs"],
    )
    nerf_coarse.load_state_dict(nerf_coarse_cfg["weights"])
    nerf_coarse = nerf_coarse.to(device)
    nerf_coarse.eval()

    nerf_fine = None
    if checkpoint["nerf_fine"] is not None:
        nerf_fine_cfg = checkpoint["nerf_fine"]
        nerf_fine = NeRF(
            depth=nerf_fine_cfg["depth"],
            W=nerf_fine_cfg["W"],
            in_channels=nerf_fine_cfg["in_channels"],
            in_channel_views=nerf_fine_cfg["in_channel_views"],
            out_channels=nerf_fine_cfg["out_channels"],
            skip_connections=nerf_fine_cfg["skip_connections"],
            use_viewdirs=nerf_fine_cfg["use_viewdirs"],
        )
        nerf_fine.load_state_dict(nerf_fine_cfg["weights"])
        nerf_fine = nerf_fine.to(device)
        nerf_fine.eval()

    return {
        "nerf_coarse": nerf_coarse,
        "nerf_fine": nerf_fine,
        "embed_pts": embed_pts,
        "embed_viewdirs": embed_viewdirs,
        "render_poses": checkpoint["render_poses"].to(device),
        "n_samples": checkpoint["n_samples"],
        "n_importance": checkpoint["n_importance"],
        "batch_size": checkpoint["batch_size"],
        "hwf": checkpoint["hwf"],
        "K": checkpoint["K"].to(device),
        "chunk": checkpoint["chunk"],
        "near": checkpoint["near"],
        "far": checkpoint["far"],
        "use_white_background": checkpoint["use_white_background"],
    }

### Lego

In [ ]:
lego_nerf_checkpoint_path = os.path.join("checkpoints", "lego", "lego_200000.pth")
lego_nerf = load_checkpoint(lego_nerf_checkpoint_path, device=device)

In [ ]:
rgbs = render_path(
    nerf_coarse=lego_nerf["nerf_coarse"],
	nerf_fine=lego_nerf["nerf_fine"],
	embed_pts=lego_nerf["embed_pts"],
	embed_viewdirs=lego_nerf["embed_viewdirs"],
	render_poses=lego_nerf["render_poses"],
	n_samples=lego_nerf["n_samples"],
	n_importance=lego_nerf["n_importance"],
	batch_size=lego_nerf["batch_size"],
	hwf=lego_nerf["hwf"],
	K=lego_nerf["K"],
	chunk=lego_nerf["chunk"],
	near=lego_nerf["near"],
	far=lego_nerf["far"],
	use_white_background=lego_nerf["use_white_background"],
)

In [ ]:
buf = io.BytesIO()
imageio.mimsave(
	buf,
	(rgbs * 255).clip(0, 255).astype(np.uint8),
    format='GIF',
	fps=30,
	loop=0,
)
buf.seek(0);

In [ ]:
display(Image(data=buf.getvalue(), format='gif'))

### Chair

In [ ]:
chair_nerf_checkpoint_path = os.path.join("checkpoints", "chair", "chair_200000.pth")
chair_nerf = load_checkpoint(chair_nerf_checkpoint_path, device=device)

In [ ]:
rgbs = render_path(
    nerf_coarse=chair_nerf["nerf_coarse"],
	nerf_fine=chair_nerf["nerf_fine"],
	embed_pts=chair_nerf["embed_pts"],
	embed_viewdirs=chair_nerf["embed_viewdirs"],
	render_poses=chair_nerf["render_poses"],
	n_samples=chair_nerf["n_samples"],
	n_importance=chair_nerf["n_importance"],
	batch_size=chair_nerf["batch_size"],
	hwf=chair_nerf["hwf"],
	K=chair_nerf["K"],
	chunk=chair_nerf["chunk"],
	near=chair_nerf["near"],
	far=chair_nerf["far"],
	use_white_background=chair_nerf["use_white_background"],
)

In [ ]:
buf = io.BytesIO()
imageio.mimsave(
	buf,
	(rgbs * 255).clip(0, 255).astype(np.uint8),
    format='GIF',
	fps=30,
	loop=0,
)
buf.seek(0);

In [ ]:
display(Image(data=buf.getvalue(), format='gif'))